### **Assignment - 1**  

1.   Name - Atish Kadam
2.   Roll No - CS25MTECH14003



**Importing library**

In [ ]:
import numpy as np
np.set_printoptions(precision=10, suppress=True)

**Load Linear Programming Instance**

In [ ]:
def check_testcase(csv_path: str):
    data = np.genfromtxt(csv_path, delimiter=",", dtype=float)
    if data.ndim == 1:
        data = data.reshape(1, -1)
    n = data.shape[1] - 1
    m = data.shape[0] - 2
    if n < 0: n = 0
    if m < 0: m = 0
    z0 = data[0, :n] if data.shape[0] > 0 else np.array([])
    c = data[1, :n] if data.shape[0] > 1 else np.array([])
    A = data[2:, :n] if data.shape[0] > 2 else np.empty((0, n))
    b = data[2:, n] if data.shape[0] > 2 else np.array([])

    return A, b, c, z0, m, n

**Active Constraints Identification**

In [ ]:
def activeness(A, b, z, tol=1e-9):
    if A.size == 0:
        return np.array([], dtype=int)
    return np.where(np.abs(b - A @ z) <= tol)[0]

**Calculate Nullspace**

In [ ]:
def check_nullspace(M, tol=1e-12):
    if M.size == 0:
        return np.eye(M.shape[1])
    U, s, Vt = np.linalg.svd(M, full_matrices=True)
    r = (s > tol).sum()
    return Vt[r:].T

**Main Algorithm**
Only possible to run when we have testcase with -
    

1.   Polytope is non-degenerate.
2.   Polytope is bounded
3.   Rak of A is n
4.   Initial feasible point is given




In [ ]:
def find_objective(csv_path: str, verbose=True, tol=1e-9, max_iter=5000):
    # Loading instance from testcase
    A, b, c, z, m, n = check_testcase(csv_path)

    # Checking for no. of variables
    if n == 0:
        print("As we got value of n as 0, this means this testcase have 0 variables which leads to trivial solution\n")
        print(f"Vertex: [], f = 0.0")
        return np.array([]), [], [0.0]

    # Check feasibility of initial point
    if m > 0 and not np.all(A @ z <= b + tol):
        raise ValueError("Initial point is not feasible (violates A z <= b).")

    # Printing details of given testcase
    if verbose:
        print("Initial Problem Setup\n")
        print(f"Variables (n):   {n}")
        print(f"Constraints (m): {m}")
        print(f"Start Point (z): {z}")
        print(f"Start Value (f): {c @ z:.10f}")
        print(f"Cost Vector (c): {c}")
        print("Constraint Matrix (A):")
        print(A)
        print(f"Constraint RHS (b): {b}\n\n")

    route = [z.copy()]
    vals = [float(c @ z)]

    # Move to first vertex
    for _ in range(5 * (m + n)):
        I = activeness(A, b, z, tol)
        if len(I) >= n:
            if verbose:
                print(f"First vertex we get is: z = {z}, f = {c @ z:.10f}")
            break

        N = check_nullspace(A[I, :], tol) if len(I) > 0 else np.eye(n)
        d = N @ (N.T @ c)
        if np.linalg.norm(d) < tol:
            d = N[:, 0] if N.shape[1] > 0 else np.zeros(n)

        Ad = A @ d
        slack = b - A @ z
        alpha, j_in = np.inf, -1
        for j in range(m):
            if Ad[j] > tol:
                step = slack[j] / Ad[j]
                if step < alpha:
                    alpha = step
                    j_in = j
        # Check for degeneracy at initial point
        if len(I) > n:
            print(f"\nERROR: Polytioope is DEGENERATE.")
            print(f"Found {len(I)} active constraints at current point, but n = {n}.")
            return None
        # Check for unboundedness
        if alpha == np.inf:
            print("\nERROR: Polytope is UNBOUNDED.")
            print("No finite constraint blocks the improving direction.")
            return None

        z = z + alpha * d
        route.append(z.copy())
        vals.append(float(c @ z))

    for it in range(max_iter):
        I = activeness(A, b, z, tol)

        # Check for degeneracy: more than n active constraints
        if len(I) > n:
            print(f"\nERROR: Polytope is DEGENERATE.")
            print(f"Found {len(I)} active constraints at vertex, but n = {n}.")
            return None

        I_sel = []
        R = None
        if len(I) >= n:
            for r in I:
                if R is None:
                    R = A[[r], :]
                    I_sel = [r]
                else:
                    candidate = np.vstack([R, A[r, :]])
                    if np.linalg.matrix_rank(candidate, tol) > np.linalg.matrix_rank(R, tol):
                        R = candidate
                        I_sel.append(r)
                        if len(I_sel) == n:
                            break

        if len(I_sel) < n:
            break

        A_I = A[I_sel, :]

        # Solve for Lagrange multipliers
        lam = np.linalg.solve(A_I.T, c)

        # Check optimality
        if np.all(lam >= -tol):
            if verbose:
                print("\nOptimal vertex reached.")
            break

        # Find leaving constraint
        k = int(np.argmin(lam))
        e = np.zeros(len(I_sel))
        e[k] = 1.0

        # Compute edge direction
        d = -np.linalg.solve(A_I, e)

        # Find blocking constraint (entering constraint)
        Ad = A @ d
        slack = b - A @ z
        alpha, j_in = np.inf, -1
        for j in range(m):
            if Ad[j] > tol:
                step = slack[j] / Ad[j]
                if step < alpha:
                    alpha = step
                    j_in = j

        # Check for unboundedness in Phase 2
        if alpha == np.inf:
            print("\nERROR: Polytope is UNBOUNDED.")
            print("Objective can be increased indefinitely.")
            return None

        z = z + alpha * d
        route.append(z.copy())
        vals.append(float(c @ z))

        if verbose:
            print(f"Entered constraint {j_in}; z={z}, f={c @ z:.10f}")

    if verbose:
        print(f"Final optimal vertex: {z}")
        print(f"Optimal value: {c @ z:.10f}")
        print("\nSequence of vertices and objective values:")
        for i, (zi, fi) in enumerate(zip(route, vals), start=1):
            print(f"{i}. {zi.tolist()}   f = {fi:.10f}")
        return None

    return z, route, vals

**Run Solver**

In [ ]:
find_objective("/content/Testcase6.csv", verbose=True)

Initial Problem Setup

Variables (n):   2
Constraints (m): 4
Start Point (z): [1. 1.]
Start Value (f): 7.0000000000
Cost Vector (c): [4. 3.]
Constraint Matrix (A):
[[1. 0.]
 [0. 1.]
 [1. 1.]
 [2. 1.]]
Constraint RHS (b): [4. 5. 6. 9.]


First vertex we get is: z = [3. 3.], f = 21.0000000000

Optimal vertex reached.
Final optimal vertex: [3. 3.]
Optimal value: 21.0000000000

Sequence of vertices and objective values:
1. [1.0, 1.0]   f = 7.0000000000
2. [3.1818181818181817, 2.6363636363636362]   f = 20.6363636364
3. [3.0, 2.9999999999999996]   f = 21.0000000000
